# POC segmentation CT cardiaque — exécution dans Colab

Ce notebook exécute les scripts du dépôt GitHub sur le GPU de Colab. Les données vont dans ton Google Drive.

**Avant de commencer** : menu **Exécution → Modifier le type d'exécution → GPU T4**. Puis exécute les cellules une par une (▶).

## 1. Relier Drive et vérifier le GPU
Une fenêtre d'autorisation s'ouvre : accepte avec ton **compte Google personnel**. Tu dois voir **Tesla T4**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Récupérer le dépôt GitHub
Le dépôt GitHub `Cameronprr/poc-segmentation-ct` est copié dans Colab (cette copie disparaît à la fin de la session ; le dépôt sur GitHub reste la référence).

In [ ]:
%cd /content
# On sort d'abord de l'ancien dossier du dépôt : sinon, après sa suppression, le dossier courant
# n'existe plus et le shell affiche l'erreur « getcwd ».
PSEUDO = 'Cameronprr'
!rm -rf /content/poc-segmentation-ct
!git clone https://github.com/{PSEUDO}/poc-segmentation-ct.git /content/poc-segmentation-ct
%cd /content/poc-segmentation-ct
!ls

## 3. Installer les bibliothèques de l'étape
Pour l'étape 2, seule `nibabel` (lecture des fichiers NIfTI) est nécessaire. Les autres (MONAI, nnU-Net, TotalSegmentator) seront installées aux étapes suivantes.

In [ ]:
!pip -q install nibabel

## 4. Étape 2 — Télécharger MM-WHS dans Drive
À ne faire qu'une fois : les données restent ensuite dans ton Drive. `megatools` est un petit outil qui télécharge depuis le service MEGA. Compte plusieurs minutes.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
!mkdir -p "$DONNEES"
!apt-get -qq install megatools > /dev/null
!megadl --path "$DONNEES" "https://mega.nz/folder/UNMF2YYI#1cqJVzo4p_wESv9P_pc8uA"

# Décompresser d'éventuelles archives
import glob, zipfile, os
for z in glob.glob(DONNEES + '/**/*.zip', recursive=True):
    print('Décompression :', z)
    zipfile.ZipFile(z).extractall(os.path.dirname(z))

## 5. Étape 2 — Vérifier les données
Le script `scripts/02_verifier_donnees.py` contrôle que les 20 cas du découpage figé sont complets et affiche leurs caractéristiques. **Résultat attendu : CONFORME — 20 cas complets.**

In [ ]:
!python scripts/02_verifier_donnees.py --donnees "$DONNEES"

**Fin de l'étape 2.** Copie le tableau affiché (ou le message d'erreur) et envoie-le dans la conversation.

## 6. Étape 3 — Installer MONAI
MONAI est une bibliothèque d'imagerie médicale (chargement des scanners, transformations, métriques). Le script de l'étape 3 s'en sert pour lire les volumes et calculer les miniatures. À refaire à chaque nouvelle session Colab, comme les cellules 1 et 2.

In [ ]:
!pip -q install "monai>=1.3" nibabel

## 7. Étape 3 — Contrôle qualité détaillé et conversion au format nnU-Net
Le script `scripts/03_preparer.py` contrôle les 20 cas : forme, affine, espacement, orientation, étiquettes et doublons entre test et entraînement. Il écrit ensuite le rapport `rapport_qc_etape3.csv`. Si tout est conforme, il écrit aussi le jeu `Dataset101_MMWHS` dans `poc_ct/nnunet_raw`.

**Résultat attendu :** « Résultat du contrôle : CONFORME », puis 15 cas dans imagesTr/labelsTr et 5 dans imagesTs/labelsTs.

Si le script s'arrête, le dernier message indique pourquoi :
- **NON CONFORME** : rien n'a été écrit au format nnU-Net ;
- **existe déjà** : le jeu a déjà été écrit. Ne relance avec `--ecraser` que si tu veux vraiment le remplacer ;
- **ERREUR PENDANT LA CONVERSION** : le dossier est incomplet. Relance avec `--ecraser` une fois la cause corrigée.

Les lignes « QUASI-DOUBLON possible » ne bloquent pas : envoie-les dans la conversation pour qu'on les examine.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct/nnunet_raw'
!python scripts/03_preparer.py --donnees "$DONNEES" --sortie "$SORTIE"

### 7 bis. Reconstruire `Dataset101_MMWHS` avec le nouveau découpage (décision D20)
**Avant :** relance les sections 1, 2 et 6. La section 2 récupère le dépôt à jour, avec le nouveau `splits/mmwhs_split.json`. Sinon, le script reconstruirait l'**ancien** découpage sans rien signaler. Pour vérifier, `!head -12 splits/mmwhs_split.json` doit montrer en test 1004, 1010, 1011, 1013 et 1014.

Le découpage figé a été remplacé une fois (D20) : le jeu nnU-Net déjà écrit dans Drive suit l'**ancien** découpage. Cette variante relance le même script avec `--ecraser` : l'ancien dossier `Dataset101_MMWHS` est **supprimé**, puis réécrit avec le découpage de `splits/mmwhs_split.json`. Le rapport `rapport_qc_etape3.csv` est aussi remplacé.

**À n'exécuter qu'une fois, maintenant.** N'exécute pas la cellule 7 juste au-dessus : elle s'arrêterait sur « existe déjà ». Une exception : si le script affiche **ERREUR PENDANT LA CONVERSION**, le dossier est incomplet. Corrige la cause, puis relance cette même cellule.

**Résultat attendu :**
- « Résultat du contrôle : CONFORME » ;
- en test `mmwhs_1004`, `1010`, `1011`, `1013` et `1014` (5 cas dans imagesTs/labelsTs), et 15 cas dans imagesTr/labelsTr ;
- la liste des paires test/entraînement de corrélation supérieure à 0,95, à m'envoyer pour qu'on vérifie qu'aucune n'est suspecte.

Si le script s'arrête sur une règle D20 (patient à plusieurs examens en test), rien n'a été modifié dans Drive. S'il affiche NON CONFORME, `Dataset101_MMWHS` n'est pas touché, mais `rapport_qc_etape3.csv` est quand même remplacé.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct/nnunet_raw'
!python scripts/03_preparer.py --donnees "$DONNEES" --sortie "$SORTIE" --ecraser

**Fin du script 03.** Copie tout ce que le script a affiché (tableau, ressemblances, avertissements, résultat) et envoie-le dans la conversation pour la vérification par l'agent QA.

## 8. Étape 3 — Examiner à l'œil les deux quasi-doublons possibles
**Examen fait le 8 novembre 2026, conclusion dans D20.** Depuis la reconstruction (section 7 bis), 1005 et 1020 sont en `imagesTr` : cette cellule afficherait « Fichier introuvable ». **Ne plus la relancer.**

Le script 03 a signalé deux paires (scanner de test / scanner d'entraînement) qui ont exactement la même taille et le même espacement, avec des miniatures très corrélées :
- `mmwhs_1005` (test) / `mmwhs_1012` (entraînement), corrélation 0,962 ;
- `mmwhs_1020` (test) / `mmwhs_1001` (entraînement), corrélation 0,974.

Si l'une de ces paires était **le même examen** (ou le même patient), le modèle aurait « vu » pendant l'entraînement un cas très proche du lot de test, et le verdict final serait trop optimiste. Mais attention : même taille et même espacement ne prouvent rien (beaucoup de scanners ont ces réglages), et **une corrélation de 0,96-0,97 entre miniatures est attendue entre deux thorax différents cadrés de la même façon**. Ce n'est pas un signal fort en soi : il faut regarder.

**Vocabulaire**
- **Coupe axiale** : une tranche horizontale du corps (comme une tranche de saucisson), vue depuis les pieds du patient. On y voit le cœur, les côtes en arc, le sternum devant et la colonne vertébrale derrière.
- **Coupe coronale** : une tranche verticale, vue de face (le patient debout face à toi). Souvent plus parlante pour reconnaître une silhouette : forme du thorax, du cœur, des poumons, hauteur du diaphragme.
- **Coupe du milieu** : la tranche située à mi-hauteur (axiale) ou à mi-profondeur (coronale) du volume.
- **Sens d'affichage** : celui des radiologues. Sur les deux vues, **la droite du patient est à gauche de l'image** (comme quand on fait face à quelqu'un). L'avant du patient est en haut de la coupe axiale, la tête en haut de la coupe coronale. Les deux scanners d'une paire sont affichés exactement dans le même sens.

**Ce qu'affiche la cellule** : pour chaque paire, une figure de 2 lignes (axiale en haut, coronale en bas) et 3 colonnes : scanner de test, scanner d'entraînement, puis leur **différence** (test − entraînement). Les deux scanners utilisent la même fenêtre d'affichage, de -200 à 800 HU (la même que le script 03).

**Comment lire la carte de différence** (blanc = 0 ; rouge = le test est plus clair ; bleu = l'entraînement est plus clair ; au-delà de ±500 HU la couleur sature) :
- **carte toute blanche** (au plus un léger bruit) : c'est **le même examen**, donc un doublon ;
- **carte non blanche** (contours en paires rouge/bleu) : ce sont des **examens différents**, mais **pas forcément des patients différents** : le même patient scanné un autre jour, ou à une autre phase du battement cardiaque, donnerait aussi des contours. Et un doublon simplement décalé d'une ou deux coupes, ou recadré, donnerait lui aussi des contours rouge/bleu : d'où le « meilleur décalage » ci-dessous.

**Pour juger « même patient »**, compare l'anatomie dans les deux premières colonnes : forme des vertèbres, des côtes et du sternum, calcifications (petites taches très blanches, par exemple sur l'aorte ou les coronaires), forme générale du thorax, taille du cœur.

**Les chiffres affichés sous chaque figure**
- **forme, espacement, orientation, origine** : l'origine est la position dans l'espace (en mm) du premier voxel du volume, lue dans l'en-tête du fichier. Deux examens distincts ont souvent des origines différentes ; une origine identique au centième de mm est un indice de copie (pas une preuve).
- **différence absolue moyenne** (axiale, coronale) : moyenne de |test − entraînement| sur la coupe affichée, en HU. 0,0 veut dire des coupes identiques au voxel près **dans la fenêtre affichée** (-200 à 800 HU). C'est un repère : l'air autour du corps vaut -200 dans les deux images (différence nulle), ce qui fait baisser la moyenne.
- **meilleur décalage** : la coupe axiale du milieu du test est comparée aux 21 coupes de l'entraînement situées de 10 coupes en dessous à 10 coupes au-dessus. On affiche le décalage qui donne la plus petite différence, et cette différence. Un doublon décalé donnerait une différence proche de 0 pour un décalage non nul ; deux examens différents restent loin de 0 quel que soit le décalage.
- **volumes entiers identiques ?** : on calcule l'empreinte (SHA-256, une suite de 64 caractères tirée de tous les voxels) de chaque volume. « IDENTIQUES » = même volume au voxel près (doublon exact) ; « différents » = au moins un voxel diffère. Le script 03 a déjà fait ce contrôle ; il est refait ici pour avoir tous les éléments au même endroit.

**Avant de lancer** : après un redémarrage de session, relance la section 1 (Drive) ; nibabel et matplotlib sont préinstallés dans Colab. La cellule ne fait que **lire** deux scanners par paire, un à la fois : elle n'écrit aucun fichier, ne lit aucun masque (le dossier `labelsTs` reste fermé, décision D18) et ne touche pas au découpage figé.

Envoie ta conclusion dans la conversation ; la décision (garder, ou retirer un cas) sera tracée dans le journal.

In [ ]:
# Comparaison à l'œil des deux paires signalées par le script 03. LECTURE SEULE : rien n'est écrit.
# Seuls les scanners (images) sont lus : aucun masque, et surtout pas labelsTs (décision D18).
import hashlib, os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

DATASET = '/content/drive/MyDrive/poc_ct/nnunet_raw/Dataset101_MMWHS'
# (numéro du cas de test, numéro du cas d'entraînement, corrélation des miniatures donnée par le script 03)
PAIRES = [('1005', '1012', 0.962),
          ('1020', '1001', 0.974)]
FENETRE = (-200, 800)   # fenêtre d'affichage en HU, identique pour les deux scanners (comme le script 03)
ECHELLE_DIFF = 500      # carte de différence de -500 à +500 HU (au-delà, la couleur sature)
DECALAGE_MAX = 10       # recherche du meilleur décalage : coupes k-10 à k+10 de l'entraînement

def vue_radiologique(coupe):
    """Écrête la coupe dans la FENETRE et la met dans le sens des radiologues pour imshow(origin='lower') :
    np.flip sur le 1er axe = droite du patient à gauche de l'image ; .T = avant (ou tête) en haut."""
    return np.flip(np.clip(coupe.astype(np.float32), *FENETRE), 0).T

def lire(chemin, k_ref=None):
    """Lit UN scanner. Renvoie ses coupes du milieu (axiale, coronale), quelques infos, et, si k_ref est
    donné, les coupes axiales k_ref-10..k_ref+10 (pour le meilleur décalage). Le volume complet n'existe
    que dans cette fonction : la mémoire est libérée à la sortie."""
    img = nib.load(chemin)
    # Remise en orientation RAS (1er axe vers la droite du patient, 2e vers l'avant, 3e vers le haut) :
    # ainsi les deux scanners d'une paire sont toujours traités dans le même sens.
    ras = nib.as_closest_canonical(img)
    vol = np.asanyarray(ras.dataobj)                              # voxels en HU
    h = hashlib.sha256(f'{vol.shape}{vol.dtype}'.encode())        # empreinte de TOUT le volume
    h.update(np.ascontiguousarray(vol).tobytes())
    sx, sy, sz = np.sqrt((ras.affine[:3, :3] ** 2).sum(axis=0))   # espacement en mm le long de chaque axe
    infos = {'forme': img.shape,
             'espacement': tuple(round(float(z), 3) for z in img.header.get_zooms()[:3]),
             'orientation': ''.join(nib.aff2axcodes(img.affine)),
             'origine': tuple(round(float(v), 2) for v in img.affine[:3, 3]),   # position du 1er voxel (mm)
             'empreinte': h.hexdigest()}
    k, j = vol.shape[2] // 2, vol.shape[1] // 2   # milieu en hauteur (axiale), milieu avant-arrière (coronale)
    # aspect = rapport des espacements, pour ne pas écraser l'image si les voxels ne sont pas cubiques.
    vues = {'axiale':   (vue_radiologique(vol[:, :, k]), k, vol.shape[2], sy / sx),
            'coronale': (vue_radiologique(vol[:, j, :]), j, vol.shape[1], sz / sx)}
    voisines = {}
    if k_ref is not None:
        for d in range(-DECALAGE_MAX, DECALAGE_MAX + 1):
            if 0 <= k_ref + d < vol.shape[2]:
                voisines[d] = vue_radiologique(vol[:, :, k_ref + d])
    del vol, ras, img
    return vues, infos, voisines

for test, entr, r in PAIRES:
    ch_test = f'{DATASET}/imagesTs/mmwhs_{test}_0000.nii.gz'
    ch_entr = f'{DATASET}/imagesTr/mmwhs_{entr}_0000.nii.gz'
    manquants = [c for c in (ch_test, ch_entr) if not os.path.exists(c)]
    if manquants:
        print('Fichier introuvable (Drive est-il relié ? section 1) :', manquants)
        continue
    v_t, i_t, _ = lire(ch_test)                                   # un volume à la fois
    v_e, i_e, voisines = lire(ch_entr, k_ref=v_t['axiale'][1])
    lus = [('TEST', f'mmwhs_{test}', v_t, i_t), ('entraînement', f'mmwhs_{entr}', v_e, i_e)]

    fig, axes = plt.subplots(2, 3, figsize=(16, 11))
    fig.suptitle(f'mmwhs_{test} (TEST) contre mmwhs_{entr} (entraînement) — corrélation des miniatures {r}'
                 '\n(droite du patient à gauche de l\'image)', fontsize=14)
    ecarts = {}
    for ligne, vue in enumerate(['axiale', 'coronale']):
        for col, (lot, nom, vues, _) in enumerate(lus):
            coupe, num, total, aspect = vues[vue]
            axes[ligne, col].imshow(coupe, cmap='gray', vmin=FENETRE[0], vmax=FENETRE[1],
                                    origin='lower', aspect=aspect)
            axes[ligne, col].set_title(f'{nom} — lot {lot}\ncoupe {vue} n°{num + 1} sur {total}')
        c_t, num_t, total_t, aspect_t = v_t[vue]                  # infos du TEST pour la carte de différence
        c_e, aspect_e = v_e[vue][0], v_e[vue][3]
        ax = axes[ligne, 2]
        if c_t.shape == c_e.shape and np.isclose(aspect_t, aspect_e):
            diff = c_t - c_e
            ecarts[vue] = float(np.abs(diff).mean())
            im = ax.imshow(diff, cmap='RdBu_r', vmin=-ECHELLE_DIFF, vmax=ECHELLE_DIFF,
                           origin='lower', aspect=aspect_t)       # RdBu_r : bleu < 0 < rouge, blanc = 0
            fig.colorbar(im, ax=ax, shrink=0.8, label='test − entraînement (HU)')
            ax.set_title(f'Différence test − entraînement\ncoupe {vue} n°{num_t + 1} sur {total_t}')
        else:
            ax.text(0.5, 0.5, 'Formes ou espacements différents :\ndifférence non calculable',
                    ha='center', va='center')
    for ax in axes.ravel():
        ax.axis('off')   # pas de graduations : seules les images comptent
    plt.tight_layout()
    plt.show()

    # Quelques chiffres simples sous la figure
    for lot, nom, _, infos in lus:
        print(f"{nom:<12} lot {lot:<13} forme {infos['forme']}  espacement {infos['espacement']} mm  "
              f"orientation {infos['orientation']}  origine {infos['origine']} mm")
    for vue, e in ecarts.items():
        print(f'Différence absolue moyenne, coupe {vue} (fenêtre {FENETRE[0]} à {FENETRE[1]} HU) : {e:.1f} HU')
    # Meilleur décalage : coupe axiale du milieu du test contre les coupes voisines de l'entraînement
    ref = v_t['axiale'][0]
    par_decalage = {d: float(np.abs(ref - c).mean()) for d, c in voisines.items() if c.shape == ref.shape}
    if par_decalage:
        d = min(par_decalage, key=par_decalage.get)
        print(f'Meilleur décalage (axiale, de -{DECALAGE_MAX} à +{DECALAGE_MAX} coupes) : {d:+d} coupe(s), '
              f'différence absolue moyenne {par_decalage[d]:.1f} HU')
    else:
        print('Meilleur décalage : non calculable (formes différentes)')
    print('Volumes entiers (empreinte SHA-256) :',
          'IDENTIQUES (doublon exact)' if i_t['empreinte'] == i_e['empreinte'] else 'différents')
    print()
    del lus, v_t, v_e, voisines

## 9. Découpage par patient (décision D20)
**Pourquoi.** L'examen de la section 8 a montré que les paires `1005`/`1012` et `1020`/`1001` ont la même forme, le même espacement et la même origine au centième de mm, mais des voxels différents : très probablement **le même patient scanné à deux phases du battement cardiaque** (cœur contracté ou relâché). Or `1005` et `1020` sont en test, `1012` et `1001` en entraînement. Un même patient des deux côtés fausse le verdict : le modèle aurait déjà « vu » ce patient. On refait donc le découpage **par patient** : le lot de test ne contient que des **patients à un seul examen** (5 cas = 5 patients indépendants, chaque patient compte une fois dans le verdict) ; les patients à plusieurs examens vont, entiers, en entraînement.

**Ce que fait le script `scripts/03b_regrouper_patients.py`.**
1. Il lit uniquement l'**en-tête** des 20 scanners (taille, espacement, position, orientation) : c'est instantané, aucun voxel n'est chargé et aucun masque n'est lu.
2. Il regroupe en « patients probables » les cas qui ont la même forme, le même espacement et la même **affine** (la matrice qui place le volume dans l'espace, origine comprise) à 0,01 mm près, et affiche tous les groupes.
3. Il vérifie que la position est **réellement enregistrée** dans l'en-tête : les colonnes `sform/qform` donnent deux codes qui le disent. S'ils valent tous les deux 0, il n'y a pas de position enregistrée : nibabel en fabrique une par défaut, identique pour tous les volumes de même taille. Le cas est alors marqué **« origine NON FIABLE »** et n'est regroupé avec aucun autre.
4. Il liste pour information les **paires proches** : même plan de coupe et même position gauche-droite et avant-arrière (à 1 mm près), mais un nombre de coupes ou une hauteur différents. Elles ne sont pas regroupées automatiquement.
5. Il tire au sort, avec la graine 42, 5 patients **à un seul examen** (et à origine fiable) pour le lot de test. Le tirage est **stratifié par épaisseur de coupe** (ici l'espacement entre deux coupes, lu dans l'en-tête) : 1 patient à 0,45 mm et 4 à 0,625 mm, pour que le test contienne les deux types d'examens. Tous les autres cas, dont les deux paires, vont en entraînement / validation croisée.
6. Il affiche la proposition, les cas qui changent de lot, et écrit le fichier `mmwhs_split_par_patient_PROPOSITION.json` dans `poc_ct` (Drive). **Le dépôt et `splits/mmwhs_split.json` ne sont pas modifiés** : la proposition y sera recopiée seulement après validation.

La géométrie seule ne prouve rien : elle s'ajoute à l'examen visuel de la section 8.

**Résultat attendu.** Au moins les groupes `{1001, 1020}` et `{1005, 1012}`, signalés « FUITE dans le découpage actuel », puis un lot de test de 5 cas (1 à 0,45 mm, 4 à 0,625 mm) qui ne contient **aucun** de ces quatre cas. Normalement, `sform/qform` n'affiche jamais `0/0`. À examiner ensemble, s'ils apparaissent :
- une ligne « ATTENTION » (groupe de plus de 2 cas, ou paire proche à cheval sur test et entraînement dans la proposition) ;
- le cadre de points d'exclamation **« ORIGINE NON FIABLE »** à la toute fin ;
- un « ARRÊT » : épaisseur de coupe imprévue (ni 0,45 ni 0,625 mm), 3e axe qui n'est pas l'axe des coupes, ou une épaisseur sans assez de patients à un seul examen pour remplir ses places (par exemple aucun à 0,45 mm). Dans ce cas, rien n'est écrit.

**Ensuite.** Copie **toute** la sortie de la cellule, y compris le contenu JSON affiché à la fin, et envoie-la dans la conversation. La reconversion au format nnU-Net viendra seulement après validation du nouveau découpage.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct'
!python scripts/03b_regrouper_patients.py --donnees "$DONNEES" --sortie "$SORTIE"

## 10. Étape 4 — Ligne de base TotalSegmentator (prédictions seulement, D19)
**Avant :** après un redémarrage de session, relance les sections 1 (Drive + GPU) et 2 (dépôt à jour). Le GPU doit être un **Tesla T4** : sans GPU, le script s'arrête (code 5) au lieu de tourner des heures sur le processeur.

**Ce que fait l'étape.** TotalSegmentator est un outil public déjà entraîné : on l'utilise tel quel, sans l'entraîner, sur les **5 scanners de test** (`imagesTs` : 1004, 1010, 1011, 1013, 1014). Sa tâche `heartchambers_highres` segmente 7 structures cardiaques, proches des 7 de MM-WHS. C'est la **ligne de base** : le score que nos modèles de l'étape 5 devront battre. Le script `scripts/04_totalsegmentator.py` :
- vérifie que `imagesTs` contient exactement les 5 scanners du découpage figé ;
- écrit une prédiction par scanner (`mmwhs_XXXX.nii.gz`) dans `poc_ct/predictions/totalsegmentator`, **hors de `nnunet_raw`** : un masque produit par TotalSegmentator ne doit jamais pouvoir servir à l'entraînement (licence, règle 5) ;
- écrit `tracabilite_etape4.json` (versions, tâche, options, date, durées, empreintes des 5 scanners).

**Aucun score n'est calculé avant l'étape 6** : le script ne lit aucun masque de vérité terrain (`labelsTs` reste fermé, D18) et ne fait aucune comparaison (D19).

**Licence.** La tâche `heartchambers_highres` n'est pas en libre accès : il faut une **clé de licence**, gratuite pour un usage non commercial, à demander ici : https://backend.totalsegmentator.com/license-academic/ (la clé ressemble à `aca_` suivi de chiffres). **Lis les conditions affichées par le formulaire avant de demander la clé** : vérifie que ton usage (projet personnel, non commercial, non clinique) est bien couvert, et en cas de doute écris à l'auteur avant de continuer. Sans clé, le script s'arrête (code 4) sans rien écrire.

Pour saisir la clé **sans l'écrire dans le notebook** :
- **le mieux** : icône **clé** (Secrets) dans la barre de gauche de Colab → ajouter un secret nommé `TOTALSEG_LICENCE`, valeur = ta clé, et autoriser l'accès du notebook ;
- **sinon** : la cellule te la demande dans une case masquée (`getpass`).

La clé passe au script par une variable d'environnement (jamais en argument de commande). Elle n'est jamais affichée ni enregistrée dans le dépôt, dans Drive ou dans la traçabilité ; TotalSegmentator la garde seulement dans la machine Colab, effacée en fin de session.

**Ce qui part vers les serveurs de TotalSegmentator.**
- La vérification de la licence et le téléchargement des poids du modèle cardiaque **transmettent forcément ta clé** au serveur de TotalSegmentator : c'est ainsi que fonctionne la licence. Aucune image n'est envoyée.
- Les **statistiques d'usage** (tâche, version, machine… et ton numéro de licence) sont envoyées par défaut par TotalSegmentator après chaque prédiction. Comme elles contiennent le numéro de licence, ce n'est pas anonyme : le script les **désactive par défaut**. L'option `--statistiques-usage` les réactive si tu le souhaites.
- Au premier lancement, TotalSegmentator affiche « TotalSegmentator sends anonymous usage statistics… » **même quand l'envoi est désactivé** : ce message vient du paquet et peut être ignoré. La ligne du script « Statistiques d'usage envoyées à TotalSegmentator : non » fait foi.

**Installation.** Surveille les avertissements de pip : l'installation peut changer la version de paquets déjà présents dans Colab (par exemple numpy, ou installer nnunetv2). Si pip affiche des conflits ou demande de redémarrer la session, envoie le message dans la conversation avant de lancer la cellule suivante.

**Durée et mémoire.** Au premier lancement de chaque session, TotalSegmentator télécharge ses poids (le modèle de recadrage, puis le modèle cardiaque via le serveur de licence). Pour ne pas saturer la mémoire vive de Colab (~12 Go), le script limite nnU-Net à **1 processus enfant** (option `--processus`, défaut 1 ; TotalSegmentator en lancerait 6 pour l'enregistrement). C'est un peu plus lent : compte quelques minutes par scanner (estimation, à confirmer).

**Si un message parle de mémoire pleine** (« RAM was full », « Background workers died ») ou si Colab redémarre tout seul : menu **Exécution → Redémarrer la session**, relance les sections 1, 2 et la cellule d'installation ci-dessous, puis la cellule de lancement en ajoutant `--ecraser` à la fin de la ligne `!python ...` (des prédictions partielles peuvent exister). Envoie-moi le message d'erreur.

**Résultat attendu :** « Lot de test vérifié : 5 scanners », « Statistiques d'usage envoyées à TotalSegmentator : non », puis pour chaque cas une ligne « forme identique au scanner : oui ; affine identique : oui ; étiquettes [1, 2, 3, 4, 5, 6, 7] », et enfin « Contrôle des prédictions : CONFORME ». Si le script s'arrête, le dernier message dit pourquoi (codes expliqués en tête du script) :
- **existe déjà** (code 2) : ne relance avec `--ecraser` que si tu veux vraiment remplacer les prédictions ;
- **CONTRÔLE EN ÉCHEC** (code 6) : les prédictions sont écrites mais une géométrie diffère ou une structure manque ; on l'examinera ensemble.

Copie **toute** la sortie de la cellule d'installation et de la cellule de lancement (la clé n'y apparaît pas) et envoie-la dans la conversation pour la vérification par l'agent QA.

In [ ]:
# Installation de TotalSegmentator, version épinglée (la même à chaque session, pour la traçabilité).
# Il apporte nnU-Net v2 et réutilise le PyTorch déjà installé dans Colab. À refaire à chaque nouvelle session.
!pip -q install "TotalSegmentator==2.18.0"
!pip show TotalSegmentator | head -2

In [ ]:
import os, getpass
IMAGES = '/content/drive/MyDrive/poc_ct/nnunet_raw/Dataset101_MMWHS/imagesTs'
SORTIE = '/content/drive/MyDrive/poc_ct/predictions/totalsegmentator'   # HORS de nnunet_raw (règle 5)

# Clé de licence : lue dans les secrets Colab (nom TOTALSEG_LICENCE), sinon demandée dans une case masquée.
# Elle passe au script UNIQUEMENT par une variable d'environnement (jamais en argument : un argument serait
# visible dans le code de la cellule et dans la liste des processus). Ni le notebook ni le dépôt ne la contiennent.
# Les statistiques d'usage de TotalSegmentator sont coupées par le script (option --statistiques-usage pour les réactiver).
try:
    from google.colab import userdata
    os.environ['TOTALSEG_LICENCE'] = userdata.get('TOTALSEG_LICENCE')
except Exception:
    os.environ['TOTALSEG_LICENCE'] = getpass.getpass('Clé de licence TotalSegmentator (aca_...) : ')

!python scripts/04_totalsegmentator.py --images "$IMAGES" --sortie "$SORTIE"
os.environ.pop('TOTALSEG_LICENCE', None)   # on retire la clé de la mémoire du notebook

## 11. Avant l'étape 6 — Comparer les définitions sur deux cas d'entraînement
**Pourquoi.** TotalSegmentator et MM-WHS numérotent leurs 7 structures de la même façon (1 = myocarde, …, 6 = aorte, 7 = artère pulmonaire : voir `docs/correspondance_etiquettes.md`), mais **ne les définissent pas forcément de la même façon**. Exemple : MM-WHS ne garde que l'**aorte ascendante**, de la valve jusqu'au bord supérieur des oreillettes ; TotalSegmentator segmente peut-être aussi la crosse ou l'aorte descendante. Comparer ces deux masques à l'étape 6 pénaliserait TotalSegmentator pour une différence de **définition**, pas pour une erreur. Les règles de comparaison de l'étape 6 doivent donc être fixées **avant** tout score, et **sans regarder le lot de test**.

**Ce que fait la section.** On regarde les écarts de définition sur **deux cas d'entraînement**, un de chaque famille d'examens : `mmwhs_1003` (coupes de 0,45 mm) et `mmwhs_1002` (0,625 mm). Le script `scripts/04b_comparer_definitions.py` :
- vérifie dans le découpage figé que ces deux cas sont bien en **entraînement** (un cas de test est refusé) ; il ne lit que `imagesTr` et `labelsTr`, **jamais `imagesTs` ni `labelsTs`** (D18) ;
- lance TotalSegmentator avec exactement les réglages de la section 10 (tâche `heartchambers_highres`, statistiques d'usage coupées, GPU obligatoire, 1 processus enfant) ;
- écrit les prédictions dans `poc_ct/predictions/totalsegmentator_definitions`, **hors de `nnunet_raw`**, sous un nom qui ne peut pas être confondu avec un masque d'entraînement (`mmwhs_1003_totalsegmentator.nii.gz`). Règle 5 : un masque produit par TotalSegmentator ne sert **jamais** à entraîner ;
- calcule quelques chiffres simples et dessine une figure par cas (voir plus bas). **Aucun score** (ni Dice, ni HD95) : on regarde les définitions, pas la performance.

Si les prédictions existent déjà dans Drive (cellule déjà lancée), elles sont **réutilisées** : seules les figures et les chiffres sont refaits, en une ou deux minutes, sans licence. Pour refaire les prédictions, ajouter `--ecraser` à la fin de la ligne `!python ...`.

**Avant de lancer.** Après un redémarrage de session : relance la section 1 (Drive + GPU), la section 2 (dépôt à jour), puis la **première cellule de la section 10** (installation de TotalSegmentator). La clé de licence est demandée comme en section 10 : secret Colab `TOTALSEG_LICENCE` (le mieux), sinon une case masquée. Elle n'est jamais affichée ni enregistrée. Durée : l'étape 4 a pris 576 s pour 5 scanners ; compte quelques minutes par scanner (1003 a des coupes plus fines, donc plus de coupes).

**Comment lire les figures.** Une figure par cas, une ligne par structure (1 à 7), le volume des deux sources dans le titre de la ligne. Cinq vues par ligne :
- **axiale, coronale, sagittale** : le scanner en gris, recadré autour de la structure (marge de 2 cm). **Contour bleu épais = vérité MM-WHS** ; **contour orange fin = TotalSegmentator**. Là où les deux coïncident, on voit un trait orange bordé de bleu. Les coupes passent par le centre de la structure de la vérité MM-WHS : on prend la coupe la plus proche de ce centre qui contient les deux masques. (Le centre de l'union des deux masques tomberait, pour l'aorte, entre l'aorte ascendante et l'aorte descendante.)
- **projection coronale (de face)** et **projection sagittale (de profil)** : le masque 3D est « écrasé » sur un plan, comme une ombre ; on voit d'un coup toute l'étendue de la structure. **Gris clair = les deux** ; **bleu = MM-WHS seul** ; **orange = TotalSegmentator seul**. Les traits pointillés marquent le niveau le plus haut et le plus bas de chaque source (bleu MM-WHS, orange TotalSegmentator).
- **Sens d'affichage** (lettres jaunes) : celui des radiologues, comme en section 8. **D/G** = droite/gauche du patient (la droite du patient est à **gauche** de l'image), **A/P** = avant/arrière, **H/B** = haut (tête)/bas (pieds). En sagittale, l'avant est à gauche.

**Que chercher, structure par structure.**
- **6 aorte** : de l'orange sous le bleu (aorte descendante, crosse) et « 2 morceaux » côté TotalSegmentator.
- **7 artère pulmonaire** : de l'orange au-delà de la bifurcation (début des branches droite et gauche).
- **1 myocarde** : un anneau orange autour du ventricule droit voudrait dire que TotalSegmentator inclut aussi la paroi du VD.
- **2 oreillette gauche** : départ des veines pulmonaires, auricule. **4 oreillette droite** : limite avec les veines caves (en haut et en bas). **5 ventricule droit** : paroi incluse ou non, chambre de chasse vers l'artère pulmonaire.
- **3 cavité du VG** : on s'attend à des contours proches ; c'est le témoin.

**Les chiffres** (tableau affiché par le script, et fichier `chiffres_definitions.csv`) :
- **vol MM / vol TS** : volume de la structure en mL (1 mL = 1 cm³) pour MM-WHS et TotalSegmentator ;
- **bas / haut** : niveau (en mm) de la coupe axiale la plus basse et la plus haute où la structure est présente ; un nombre plus grand est plus près de la tête ;
- **écart** = TotalSegmentator − MM-WHS, en mm. **Écart « bas » négatif** = TotalSegmentator descend plus bas ; **écart « haut » positif** = il monte plus haut ;
- **morceaux** : nombre de blocs séparés de chaque masque.

Deux cas, c'est une **indication**, pas une mesure : un écart qui apparaît sur les deux cas est probablement une différence de définition ; un écart sur un seul cas peut être une erreur ponctuelle.

**Si le script s'arrête**, le dernier message dit pourquoi : 1 = garde-fou (rien n'est écrit), 2 = prédictions incomplètes déjà présentes (relancer avec `--ecraser`), 3 = erreur pendant la prédiction, 4 = licence, 5 = pas de GPU, 6 = géométrie différente entre scanner, vérité et prédiction (la figure du cas fautif et le tableau ne sont pas faits ; la figure d'un cas traité avant peut déjà exister).

**Lire les niveaux en mm.** La valeur absolue d'un niveau dépend du repère de chaque scanner : seuls les **écarts** TotalSegmentator − MM-WHS (calculés dans le même repère pour les deux masques) sont à interpréter.

**Ce qu'il faut m'envoyer** : toute la sortie de la cellule de lancement (la clé n'y apparaît pas), les deux tableaux de chiffres, et les deux figures (captures d'écran de la dernière cellule, ou les fichiers `definitions_mmwhs_1003.png` et `definitions_mmwhs_1002.png` de Drive), ou à défaut ta description, structure par structure. Avec ces éléments, on fixera les règles de comparaison de l'étape 6 (par exemple : rapporter à part les structures définies différemment, limiter la comparaison à une zone fixée d'avance, ou ne comparer que les 4 cavités et le myocarde). Ces règles seront écrites dans le journal des décisions **avant** tout calcul de score.

In [ ]:
import os, getpass
DATASET = '/content/drive/MyDrive/poc_ct/nnunet_raw/Dataset101_MMWHS'   # seuls imagesTr et labelsTr sont lus
SORTIE = '/content/drive/MyDrive/poc_ct/predictions/totalsegmentator_definitions'   # HORS de nnunet_raw (règle 5)

# Clé de licence : comme en section 10. Lue dans les secrets Colab (nom TOTALSEG_LICENCE), sinon demandée dans
# une case masquée. Elle passe au script UNIQUEMENT par une variable d'environnement, jamais en argument.
# Inutile si les prédictions existent déjà (elles sont réutilisées) : la case peut alors rester vide (Entrée).
try:
    from google.colab import userdata
    os.environ['TOTALSEG_LICENCE'] = userdata.get('TOTALSEG_LICENCE')
except Exception:
    os.environ['TOTALSEG_LICENCE'] = getpass.getpass('Clé de licence TotalSegmentator (aca_...) : ')

# Deux cas d'ENTRAÎNEMENT, un par famille d'examens : 1003 (0,45 mm) et 1002 (0,625 mm).
!python scripts/04b_comparer_definitions.py --dataset "$DATASET" --sortie "$SORTIE" --cas 1003 1002
os.environ.pop('TOTALSEG_LICENCE', None)   # on retire la clé de la mémoire du notebook

In [ ]:
# Affiche les figures et les chiffres écrits par le script dans Drive. LECTURE SEULE : rien n'est recalculé.
import os
import pandas as pd
from IPython.display import Image, display

SORTIE = '/content/drive/MyDrive/poc_ct/predictions/totalsegmentator_definitions'
for n in ('1003', '1002'):
    figure = f'{SORTIE}/definitions_mmwhs_{n}.png'
    if os.path.exists(figure):
        print(f'mmwhs_{n} — contours : bleu = vérité MM-WHS, orange = TotalSegmentator')
        display(Image(filename=figure))
    else:
        print(f'Figure introuvable : {figure} (lancer d\'abord la cellule précédente)')

chiffres = f'{SORTIE}/chiffres_definitions.csv'
if os.path.exists(chiffres):
    # vol_ml = volume (mL) ; bas_mm / haut_mm = niveaux extrêmes (mm) ; écart = TotalSegmentator − MM-WHS
    display(pd.read_csv(chiffres))

## 12. Étape 5 — Installer nnU-Net v2 et MLflow
**Avant :** après un redémarrage de session, relance les sections 1 (Drive + GPU) et 2 (dépôt à jour).

**nnU-Net v2** est la « recette » d'entraînement de référence en segmentation médicale : il choisit lui-même ses réglages (taille des morceaux d'image, architecture du réseau, augmentations) à partir des données. On l'installe en version **2.8.1**, épinglée. C'est la version que pip avait choisie à l'étape 4 en installant TotalSegmentator 2.18.0 (qui demande seulement `nnunetv2` 2.3.1 ou plus récent).

**MLflow** est un carnet de bord des entraînements : pour chaque essai, il range les paramètres, les mesures (durée des époques, pertes, Dice) et les fichiers (journal, courbes). Ici, tout est stocké **dans ton Drive** (`poc_ct/mlflow`), sans serveur. La version légère `mlflow-skinny` suffit ; `alembic` et `sqlalchemy` lui servent à écrire sa base de données (un fichier SQLite). Leurs versions sont épinglées aussi ; `sqlalchemy` remplace la version préinstallée dans Colab, sans effet attendu sur le reste du notebook.

**À surveiller : la version de PyTorch.** PyTorch est la bibliothèque de calcul sur GPU utilisée par nnU-Net. nnU-Net 2.8.1 refuse PyTorch 2.9. Si Colab fournit cette version, pip téléchargera une autre version de PyTorch, qui peut ne pas correspondre à la version de CUDA (le pilote de calcul du GPU) installée dans Colab. On n'épingle pas PyTorch nous-mêmes : la bonne version dépend de ce CUDA, qui change d'une mise à jour de Colab à l'autre. À la place, la cellule compare la version avant et après l'installation :
- **versions identiques** : rien à faire ;
- **« PyTorch a changé de version »** : des messages de conflit avec `torchvision` ou `torchaudio` sont alors attendus. Fais **Exécution → Redémarrer la session**, relance les sections 1 et 2 (pas cette cellule), puis envoie-moi toute la sortie **avant** de lancer la section 13 : on vérifiera ensemble que le GPU est toujours utilisable.

Envoie aussi la sortie si pip affiche d'autres conflits.

In [ ]:
# Versions épinglées (les mêmes à chaque session, pour la traçabilité). À refaire à chaque nouvelle session.
import subprocess, torch
avant = torch.__version__
print('torch avant installation :', avant)
!pip -q install "nnunetv2==2.8.1" "mlflow-skinny==3.16.1" "alembic==1.20.1" "sqlalchemy==2.1.6"
!pip show nnunetv2 mlflow-skinny alembic sqlalchemy | grep -E "^(Name|Version)"
# Version de PyTorch APRÈS installation, lue par pip (l'import déjà fait garde l'ancienne en mémoire).
infos = subprocess.run(['pip', 'show', 'torch'], capture_output=True, text=True).stdout
apres = next((l.split(': ', 1)[1] for l in infos.splitlines() if l.startswith('Version:')), 'inconnue')
print('torch après installation :', apres)
if apres != avant:
    print('!!! PyTorch a changé de version : Exécution → Redémarrer la session, puis sections 1 et 2 '
          '(pas cette cellule). Envoie cette sortie dans la conversation avant de continuer.')

## 13. Étape 5 — Préparer nnU-Net : vérification, prétraitement, plis par patient
Le script `scripts/05a_preparer_nnunet.py` fait les **vérifications prévues au début de l'étape 5** (README), puis prépare l'entraînement :
1. il vérifie que `imagesTr` et `labelsTr` contiennent exactement les 15 cas d'entraînement du découpage figé, et **aucun cas de test**. Il ne regarde jamais `imagesTs` ni `labelsTs` (D18) ;
2. il construit les **5 plis de validation croisée par patient** (un pli = un groupe d'environ 3 cas mis de côté à tour de rôle pour valider) : les paires `1001/1020` et `1005/1012` restent toujours dans le même pli (D20). Les plis sont aussi **stratifiés par épaisseur de coupe**, comme le lot de test (D20). Le jeu contient deux familles d'examens, à coupes de 0,45 mm et de 0,625 mm ; stratifier, c'est répartir chaque famille séparément pour que chaque pli ressemble au jeu entier. L'épaisseur est lue dans l'en-tête des images de `imagesTr` (même définition que le script 03b). Les 3 patients d'entraînement à 0,45 mm (`1001/1020`, `1003`, `1016`) vont dans 3 plis différents, tirés au sort ; les patients à 0,625 mm complètent ensuite les plis les moins remplis. Le tirage utilise une graine fixe (42) : on retrouve les mêmes plis à chaque fois. Si les épaisseurs lues ne sont pas celles attendues, le script s'arrête (code 1) ;
3. il compare lui-même, cas par cas, **avec SimpleITK** (la bibliothèque de lecture d'images de nnU-Net), chaque image de `imagesTr` à son masque de `labelsTr` : même taille, même espacement, même origine, même orientation, avec la même tolérance que nnU-Net. Il vérifie aussi qu'aucun masque n'est vide. **C'est ce contrôle qui fait foi pour le point A3** (README) : il ne dépend pas de ce que nnU-Net affiche ;
4. il lance `nnUNetv2_plan_and_preprocess` avec `--verify_dataset_integrity`, qui refait ces vérifications à sa façon. Le script s'arrête au moindre avertissement. La commande calcule ensuite l'« empreinte » du jeu (statistiques de taille, d'espacement, d'intensité), choisit les réglages (« plans ») et **prétraite** les images (ré-échantillonnage à un espacement commun, normalisation des intensités) ;
5. seulement si tout est conforme, il écrit un **marqueur** `verification_05a.json` (vérification réussie, date, cas vérifiés), puis `splits_final.json` (les plis que nnU-Net lira) et sa fiche de traçabilité `splits_final_05a.json`, dans `poc_ct/nnunet_preprocessed/Dataset101_MMWHS/`. Au début de chaque vérification, l'ancienne fiche et l'ancien marqueur sont effacés ; en cas d'arrêt, ils ne sont pas réécrits, et le script 05b refusera de lancer.

**Pourquoi la seule configuration `3d_fullres`.** Par défaut, nnU-Net entraîne plusieurs configurations (2D coupe par coupe, 3D pleine résolution, 3D basse résolution puis affinage), 5 plis chacune, et garde la meilleure. Sur Colab gratuit, une seule est envisageable. `3d_fullres` est celle que recommandent les auteurs de nnU-Net quand on n'en entraîne qu'une. Ne prétraiter qu'elle économise aussi du temps et de la place dans Drive.

**Mémoire.** Le script limite nnU-Net à **2 processus** (programmes lancés en parallèle ; nnU-Net en lancerait 4 à 8). Colab gratuit n'a que 2 processeurs et environ 12 Go de mémoire vive : au-delà, on ne gagne rien et la mémoire sature. Si un message parle de mémoire pleine, ou si Colab redémarre tout seul : relance les sections 1, 2 et 12, puis cette cellule avec `--processus 1` ajouté à la fin de la ligne `!python ...`.

**Durée** : inconnue, probablement plusieurs dizaines de minutes (15 scanners lus depuis Drive, ré-échantillonnés, puis écrits dans Drive).

**Résultat attendu :** les 5 plis affichés avec l'épaisseur de chaque cas, 3 cas par pli, le pli de `1001/1020` et celui de `1005/1012` contenant les deux cas de la paire, au plus un patient à 0,45 mm par pli, 15 lignes « conforme » puis « Contrôle SimpleITK : CONFORME », « verify_dataset_integrity Done », puis « aucun écart signalé », « Cas prétraités en 3d_fullres : 15 » et « Résultat : PRÊT ». Si le script s'arrête, le dernier message dit pourquoi :
- **code 4, « NE PAS ENTRAÎNER »** : le contrôle SimpleITK ou nnU-Net a vu un écart de géométrie entre une image et son masque, ou un masque vide (pour nnU-Net, ce n'est souvent qu'un avertissement : le script, lui, s'arrête). C'est le point A3 de la relecture de l'étape 3 : envoie-moi le message ;
- **code 2, « existe déjà et est DIFFÉRENT »** : un autre `splits_final.json` est déjà là, peut-être tiré au hasard par nnU-Net. N'ajoute `--ecraser` qu'après en avoir parlé ;
- **prétraitement déjà fait** (cellule relancée) : ajoute `--sauter-pretraitement` pour n'écrire que les plis, sans tout recommencer. Le contrôle SimpleITK est refait (quelques minutes), et le marqueur d'une vérification réussie est exigé : s'il manque, le script s'arrête (code 1) et il faut relancer sans cette option.

Envoie **toute** la sortie dans la conversation (le journal complet de nnU-Net est aussi dans `poc_ct/nnunet_preprocessed/journal_05a_plan_and_preprocess.txt`).

In [ ]:
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # jeu écrit à l'étape 3
PRE = '/content/drive/MyDrive/poc_ct/nnunet_preprocessed'           # données prétraitées + plis
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'                # modèles et points de reprise

!python scripts/05a_preparer_nnunet.py --raw "$RAW" --preprocessed "$PRE" --results "$RES"

## 14. Étape 5 — Essai à blanc : 5 époques sur le pli 0
**But : mesurer, pas encore entraîner pour de bon.** Le réglage par défaut de nnU-Net (1 000 époques × 5 plis) est impossible sur Colab gratuit. Avant de choisir un nombre d'époques, on mesure la durée d'une **époque** sur le T4. Pour nnU-Net, une époque = 250 pas d'entraînement sur des morceaux d'image (des « patchs ») tirés au hasard, puis 50 pas de validation rapide. On utilise la variante `nnUNetTrainer_5epochs` : identique au réglage par défaut, sauf qu'elle s'arrête après 5 époques. **Le résultat ne servira pas de modèle** : seulement à estimer les durées. Rien n'est décidé dans le script : on choisira ensemble la suite d'après les chiffres.

**Ce que fait `scripts/05b_essai_nnunet.py`.**
1. Il vérifie que le marqueur `verification_05a.json` de la section 13 est présent (vérification image/masque réussie), puis que `splits_final.json` existe et est **bien celui écrit par 05a** (les plis sont recalculés à partir du découpage figé, de la graine et des épaisseurs de coupe relues dans les en-têtes de `imagesTr`) : même empreinte que dans la fiche de traçabilité, et mêmes plis une fois recalculés depuis le découpage figé. Sans ce fichier, nnU-Net tirerait ses plis au hasard, cas par cas.
2. Il **copie les données prétraitées de Drive vers le disque local** de la machine Colab (`/content/nnunet_preprocessed_local`). Pendant l'entraînement, nnU-Net relit sans arrêt ces fichiers : depuis Drive, ce serait beaucoup plus lent, avec un risque d'erreurs de lecture. Les **résultats et points de reprise**, eux, sont écrits **dans Drive**, pour survivre à la fin de la session.
3. Il lance `nnUNetv2_train 101 3d_fullres 0 -tr nnUNetTrainer_5epochs`. À la fin, nnU-Net fait sa **validation finale** : il segmente en entier les 3 scanners de validation du pli 0 et calcule leur Dice. Ces 3 cas font partie des 15 d'entraînement : **le lot de test n'est jamais utilisé**.
4. Il lit le **journal d'entraînement** de nnU-Net (`training_log_*.txt`). Il vérifie que nnU-Net a écrit « Using splits from existing split file » avec le bon fichier, et que les 3 scanners prédits en validation finale sont bien ceux du pli 0. Il affiche pour chaque époque la durée, les pertes et le **Dice pseudo** (Dice approximatif, calculé sur quelques patchs : seulement une tendance). Il affiche aussi la durée moyenne **sans la 1re époque** (plus lente : compilation, mise en route) et une **extrapolation indicative** pour 100, 250 et 1 000 époques. La durée « Epoch time » de nnU-Net n'inclut pas l'écriture de `checkpoint_best.pth` dans Drive : le script affiche ce temps « hors époque » à part, à ajouter à l'extrapolation.
5. Il trace l'essai dans **MLflow** : paramètres (pli, trainer, configuration, versions, GPU, version du code), durées, pertes, Dice pseudo, et le journal en pièce jointe.

**Risques pratiques, et réglages proposés** (options à ajouter à la fin de la ligne `!python ...`) :
- **Lectures depuis Drive** : évitées par la copie locale (par défaut). `--lire-depuis-drive` la désactive (déconseillé).
- **torch.compile** : par défaut, nnU-Net fait traduire le réseau par PyTorch en code optimisé pour le GPU. Ensuite, c'est plus rapide, mais la compilation ajoute plusieurs minutes à la 1re époque, et elle peut échouer sur certaines combinaisons de GPU et de versions. Si l'essai plante avec un message qui parle de `torch._dynamo`, `inductor` ou `triton`, relance avec `--ecraser --sans-compile`. On garde le réglage par défaut d'abord, pour que la mesure soit représentative.
- **Mémoire vive** : nnU-Net prépare les patchs (augmentation de données) dans des processus parallèles. Il en lance au plus autant que de processeurs, soit 2 sur Colab gratuit : c'est probablement ce qui limitera la vitesse, plus que le GPU. L'export de la validation finale est limité à 2 processus (`--processus 1` si la mémoire sature).
- **Validation finale** : sa durée est mesurée à part (3 volumes entiers, à pleine résolution). Elle s'ajoutera à chaque pli de l'entraînement définitif.
- **Session coupée** : Colab gratuit coupe au bout de quelques heures, ou plus tôt si l'onglet reste inactif. nnU-Net écrit `checkpoint_best.pth` (meilleur modèle) à chaque progrès, et `checkpoint_latest.pth` seulement **toutes les 50 époques** : une coupure peut donc faire perdre jusqu'à 49 époques. Pour **reprendre** : relance les sections 1, 2 et 12, puis cette cellule avec `--continuer` (option `--c` de nnU-Net, qui repart du dernier point de reprise). Si l'entraînement s'est terminé mais que le traçage MLflow a manqué : `--tracer-seulement`.

**Durée** : c'est ce que l'essai mesure. Prévois au moins une heure de session.

**Résultat attendu :** « splits_final.json vérifié », la copie locale, la sortie de nnU-Net (5 époques), puis le tableau des époques, la durée moyenne, l'extrapolation, « Essai tracé dans MLflow » et « Résultat : ESSAI TERMINÉ ». Codes d'arrêt : 1 = garde-fou (rien n'est lancé) ; 2 = un essai existe déjà (`--continuer` ou `--ecraser`) ; 3 = nnU-Net s'est arrêté en erreur ; 5 = pas de GPU ; **6 = le journal ne confirme pas que nnU-Net a utilisé nos plis : ne pas utiliser les résultats**.

Envoie **toute** la sortie de la cellule dans la conversation, ainsi que le tableau et la courbe affichés par la cellule suivante.

In [ ]:
RAW = '/content/drive/MyDrive/poc_ct/nnunet_raw'                    # jeu écrit à l'étape 3
PRE = '/content/drive/MyDrive/poc_ct/nnunet_preprocessed'           # données prétraitées + plis
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'                # modèles et points de reprise
MLFLOW = '/content/drive/MyDrive/poc_ct/mlflow'                     # carnet de bord MLflow
!python scripts/05b_essai_nnunet.py --raw "$RAW" --preprocessed "$PRE" --results "$RES" --mlflow "$MLFLOW"

In [ ]:
# Affiche les essais enregistrés dans MLflow et la courbe d'apprentissage de nnU-Net. LECTURE SEULE.
# La base de Drive est copiée avant lecture : elle n'est jamais ouverte directement (verrous de fichiers sur Drive).
import os, shutil, tempfile
import mlflow
from IPython.display import Image, display

MLFLOW = '/content/drive/MyDrive/poc_ct/mlflow'
RES = '/content/drive/MyDrive/poc_ct/nnunet_results'
if not os.path.exists(f'{MLFLOW}/mlflow.db'):
    print(f'Aucune base MLflow dans {MLFLOW} : lancer d\'abord la cellule précédente jusqu\'à « Essai tracé dans MLflow ».')
else:
    copie = os.path.join(tempfile.mkdtemp(), 'mlflow.db')
    shutil.copy2(f'{MLFLOW}/mlflow.db', copie)
    mlflow.set_tracking_uri('sqlite:///' + copie)
    essais = mlflow.search_runs(experiment_names=['etape5_nnunet'])
    colonnes = ['start_time', 'tags.statut', 'params.trainer', 'params.pli', 'params.gpu', 'params.torch_compile',
                'metrics.nb_epoques_terminees', 'metrics.duree_1re_epoque_s', 'metrics.duree_epoque_moyenne_sans_1re_s',
                'metrics.hors_epoque_s', 'metrics.validation_finale_s', 'metrics.dice_validation_finale',
                'metrics.extrapolation_1000_epoques_h_par_pli']
    display(essais[[c for c in colonnes if c in essais.columns]])

# progress.png : courbes de nnU-Net (pertes, Dice pseudo, durée des époques, taux d'apprentissage).
courbe = f'{RES}/Dataset101_MMWHS/nnUNetTrainer_5epochs__nnUNetPlans__3d_fullres/fold_0/progress.png'
if os.path.exists(courbe):
    display(Image(filename=courbe))
else:
    print('Courbe introuvable :', courbe)